<a href="https://colab.research.google.com/github/Yesid-Valencia/Deep-Learning---Especializacion-IA/blob/main/R2_A3_S12_Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# CELDA 1: Instalación limpia
!pip install -q gradio transformers sentencepiece sacremoses easyocr

In [4]:
# CELDA 2: Traductor con AutoModel (Compatible con Transformers v5) + EasyOCR + Gradio

# Importamos PyTorch, la librería principal para cargar y ejecutar los modelos de Deep Learning
import torch

# Importamos Gradio para diseñar de manera muy sencilla la interfaz gráfica de nuestra aplicación
import gradio as gr

# Importamos EasyOCR para poder extraer texto desde las imágenes de forma rápida
import easyocr

# Importamos NumPy para manejar y transformar las imágenes en arreglos numéricos matriciales
import numpy as np

# Importamos Image de PIL para poder visualizar y manipular las imágenes cargadas por el usuario
from PIL import Image

# Importamos los componentes de Hugging Face Transformers para procesar textos y usar modelos Seq2Seq
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Configurar dispositivo (GPU si está disponible, sino CPU)
# Detectamos de forma automática si la máquina tiene tarjeta gráfica Nvidia (GPU) para ir mucho más rápido
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Ejecutando en: {device.upper()}")

# 2. Cargar Modelos y Tokenizadores oficiales (Helsinki-NLP)
# Cargamos el preparador de texto (tokenizador) para traducir de Español a Inglés
print("⏳ Cargando modelo Español ➔ Inglés...")
model_name_es_en = "Helsinki-NLP/opus-mt-es-en"
tokenizer_es_en = AutoTokenizer.from_pretrained(model_name_es_en)
# Cargamos el modelo cerebral de traducción Español-Inglés y lo mandamos al procesador configurado (GPU/CPU)
model_es_en = AutoModelForSeq2SeqLM.from_pretrained(model_name_es_en).to(device)

# Cargamos el preparador de texto (tokenizador) para traducir de Inglés a Español
print("⏳ Cargando modelo Inglés ➔ Español...")
model_name_en_es = "Helsinki-NLP/opus-mt-en-es"
tokenizer_en_es = AutoTokenizer.from_pretrained(model_name_en_es)
# Cargamos el modelo cerebral de traducción Inglés-Español y lo enviamos al procesador
model_en_es = AutoModelForSeq2SeqLM.from_pretrained(model_name_en_es).to(device)

# 3. Inicializar EasyOCR
print("⏳ Inicializando detector OCR...")
# Creamos el lector OCR configurado para leer texto en inglés y español, usando GPU si está disponible
ocr_reader = easyocr.Reader(['en', 'es'], gpu=(device == "cuda"))
print("✅ ¡Modelos listos!")

# 4. Función de traducción directa con PyTorch
def traducir_texto(texto: str, direccion: str) -> str:
    # Verificamos si el usuario ingresó un texto vacío o puros espacios en blanco
    if not texto or not texto.strip():
        return "⚠️ Por favor, ingresa un texto para traducir."

    # Dependiendo de la opción elegida por el usuario, seleccionamos el par de tokenizador y modelo correspondiente
    if direccion == "Español ➔ Inglés":
        tok, mod = tokenizer_es_en, model_es_en
    else:
        tok, mod = tokenizer_en_es, model_en_es

    # Convertimos las palabras en números legibles para el modelo y los pasamos al dispositivo (GPU o CPU)
    inputs = tok(texto, return_tensors="pt", padding=True, truncation=True).to(device)

    # Realizamos la traducción desactivando el cálculo de gradientes para ahorrar memoria y agilizar la respuesta
    with torch.no_grad():
        salida_ids = mod.generate(**inputs, max_length=512)

    # Decodificamos el resultado numérico para transformarlo de vuelta a texto común y corriente sin caracteres raros
    return tok.decode(salida_ids[0], skip_special_tokens=True)

# 5. Función de OCR + Traducción
def traducir_imagen(imagen: Image.Image, direccion: str):
    # Si no se ha cargado ninguna imagen, avisamos al usuario amablemente
    if imagen is None:
        return "⚠️ No se ha subido ninguna imagen.", ""

    # Convertimos la imagen de formato PIL a una matriz numérica de NumPy para que EasyOCR pueda procesarla
    img_np = np.array(imagen)
    # Buscamos cualquier bloque de texto que aparezca en la imagen de manera automatizada
    textos_detectados = ocr_reader.readtext(img_np, detail=0)
    # Unimos todos los fragmentos de texto detectados en un solo párrafo
    texto_extraido = " ".join(textos_detectados).strip()

    # Si EasyOCR no logró detectar ninguna letra, devolvemos un mensaje informativo
    if not texto_extraido:
        return "🔍 No se detectó texto en la imagen.", ""

    # Enviamos el texto extraído por la cámara a nuestra función anterior para que lo traduzca
    traduccion = traducir_texto(texto_extraido, direccion)
    return texto_extraido, traduccion

# 6. Interfaz Web con Gradio
# Creamos el contenedor principal de nuestra interfaz visual usando un tema visual suave (Soft)
with gr.Blocks(theme=gr.themes.Soft(), title="Traductor IA") as demo:
    # Mostramos un título principal llamativo en la parte superior
    gr.Markdown("# 🌐 Traductor Multimodal de Texto e Imágenes (EN ↔ ES)")

    # Creamos un sistema de pestañas para organizar las herramientas
    with gr.Tabs():
        # Pestaña 1: Traducción directa usando texto escrito
        with gr.TabItem("📝 Traducción de Texto"):
            # Área para que el usuario escriba su texto original
            txt_in = gr.Textbox(label="Ingresa texto", placeholder="Escribe aquí...", lines=4)
            # Botones circulares para que seleccione el sentido de la traducción
            dir_txt = gr.Radio(["Español ➔ Inglés", "Inglés ➔ Español"], value="Español ➔ Inglés", label="Dirección")
            # Botón de acción para iniciar el proceso
            btn_txt = gr.Button("Traducir Texto", variant="primary")
            # Cuadro de texto de solo lectura donde se mostrará el resultado traducido
            txt_out = gr.Textbox(label="Traducción", lines=4, interactive=False)

            # Vinculamos el clic del botón con la función para que reciba la entrada y asigne la salida
            btn_txt.click(traducir_texto, inputs=[txt_in, dir_txt], outputs=txt_out)

        # Pestaña 2: Captura de imagen con traducción posterior
        with gr.TabItem("📷 Traductor desde Imagen (OCR)"):
            # Zona interactiva para arrastrar o subir la imagen que tiene el texto
            img_in = gr.Image(type="pil", label="Sube una foto con texto en inglés")
            # Selección para indicar el idioma de origen y destino
            dir_img = gr.Radio(["Inglés ➔ Español", "Español ➔ Inglés"], value="Inglés ➔ Español", label="Dirección de traducción")
            # Botón de acción para activar el escáner
            btn_img = gr.Button("Escanear y Traducir", variant="primary")
            # Cuadro donde aparecerá la transcripción idéntica de la imagen
            ocr_out = gr.Textbox(label="1. Texto detectado por OCR", lines=2, interactive=False)
            # Cuadro donde se reflejará el texto final ya traducido
            img_tr_out = gr.Textbox(label="2. Traducción al Español", lines=3, interactive=False)

            # Vinculamos el clic para realizar la detección visual y su posterior traducción en un solo paso
            btn_img.click(traducir_imagen, inputs=[img_in, dir_img], outputs=[ocr_out, img_tr_out])

# Arrancamos la aplicación y creamos un enlace temporal de uso público para compartirlo
demo.launch(share=True)

🚀 Ejecutando en: CPU
⏳ Cargando modelo Español ➔ Inglés...


config.json:   0%|          | 0.00/1.44k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.59M [00:00<?, ?B/s]

... (18 outputs left out)

⏳ Inicializando detector OCR...
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

/usr/local/lib/python3.13/dist-packages/torch/ao/nn/quantized/dynamic/modules/rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/quantized/Quantizer.cpp:111.)
  w_ih = torch.quantize_per_tensor(
/tmp/ipykernel_832/1795114641.py:70: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Traductor IA") as demo:


✅ ¡Modelos listos!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://85de06b3d4794ce9bc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
